# Guía Práctica: Importación y Exportación de Archivos `.arff` en R

En este cuaderno aprenderemos a trabajar con archivos en formato **ARFF** (*Attribute-Relation File Format*), el estándar utilizado por herramientas de aprendizaje automático como **Weka**.

Utilizaremos como referencia la estructura clásica del dataset de Weka **`breast-cancer.arff`** y veremos dos enfoques para manipular estos archivos en **R**:
1. **Usando la librería estándar `foreign` (`read.arff` y `write.arff`)**.
2. **Escritura manual controlada** (muy útil cuando queremos garantizar que no aparezcan comillas innecesarias o problemas de formato en las cabeceras nominales).

## 1. Preparación: Crear un archivo `breast-cancer.arff` de ejemplo

Para que este cuaderno sea 100% autocontenido y puedas ejecutarlo directamente sin descargar archivos externos, primero crearemos en disco una muestra fiel del dataset original de Weka `breast-cancer.arff`.

In [1]:
# Contenido representativo del dataset clásico breast-cancer.arff de Weka
contenido_arff <- "@relation breast-cancer

@attribute age {'10-19','20-29','30-39','40-49','50-59','60-69','70-79','80-89','90-99'}
@attribute menopause {'lt40','ge40','premeno'}
@attribute tumor-size {'0-4','5-9','10-14','15-19','20-24','25-29','30-34','35-39','40-44','45-49','50-54','55-59'}
@attribute inv-nodes {'0-2','3-5','6-8','9-11','12-14','15-17','18-20','21-23','24-26','27-29','30-32','33-35','36-39'}
@attribute node-caps {'yes','no'}
@attribute deg-malig {'1','2','3'}
@attribute breast {'left','right'}
@attribute breast-quad {'left_up','left_low','right_up','right_low','central'}
@attribute irradiat {'yes','no'}
@attribute Class {'no-recurrence-events','recurrence-events'}

@data
'40-49','premeno','15-19','0-2','yes','3','right','left_up','no','recurrence-events'
'50-59','ge40','15-19','0-2','no','1','right','central','no','no-recurrence-events'
'50-59','ge40','35-39','0-2','no','2','left','left_low','no','recurrence-events'
'40-49','premeno','35-39','0-2','yes','3','right','left_low','yes','no-recurrence-events'
'40-49','premeno','30-34','3-5','yes','2','left','right_up','no','recurrence-events'
'50-59','premeno','25-29','3-5','no','2','right','left_up','yes','no-recurrence-events'
'50-59','ge40','40-44','0-2','no','3','left','left_up','no','no-recurrence-events'
'40-49','premeno','10-14','0-2','no','2','left','left_up','no','no-recurrence-events'
'40-49','premeno','0-4','0-2','no','2','right','right_low','no','no-recurrence-events'
'40-49','ge40','40-44','15-17','yes','2','right','left_up','yes','no-recurrence-events'"

# Guardamos el archivo en el directorio de trabajo
writeLines(contenido_arff, "breast-cancer.arff")
cat("Archivo 'breast-cancer.arff' creado correctamente.\n")

Archivo 'breast-cancer.arff' creado correctamente.


## 2. Importación de un archivo `.arff` a un `data.frame` en R

El paquete **`foreign`** viene preinstalado en la mayoría de distribuciones de R y proporciona la función `read.arff()`, que convierte directamente los atributos nominales del archivo ARFF en variables de tipo `factor` en R.

*(Nota: Otra alternativa moderna disponible en CRAN es el paquete `farff`, que resulta más rápido con datasets muy grandes).* 

In [2]:
# Cargamos la librería foreign
library(foreign)

# Importamos el archivo .arff
datos_cancer <- read.arff("breast-cancer.arff")

# Inspeccionamos la estructura del data.frame resultante
str(datos_cancer)

# Mostramos las primeras filas
head(datos_cancer)

'data.frame':	10 obs. of  10 variables:
 $ age        : Factor w/ 2 levels "40-49","50-59": 1 2 2 1 1 2 2 1 1 1
 $ menopause  : Factor w/ 2 levels "ge40","premeno": 2 1 1 2 2 2 1 2 2 1
 $ tumor-size : Factor w/ 7 levels "0-4","10-14",..: 3 3 6 6 5 4 7 2 1 7
 $ inv-nodes  : Factor w/ 3 levels "0-2","15-17",..: 1 1 1 1 3 3 1 1 1 2
 $ node-caps  : Factor w/ 2 levels "no","yes": 2 1 1 2 2 1 1 1 1 2
 $ deg-malig  : Factor w/ 3 levels "1","2","3": 3 1 2 3 2 2 3 2 2 2
 $ breast     : Factor w/ 2 levels "left","right": 2 2 1 2 1 2 1 1 2 2
 $ breast-quad: Factor w/ 5 levels "central","left_low",..: 3 1 2 2 5 3 3 3 4 3
 $ irradiat   : Factor w/ 2 levels "no","yes": 1 1 1 2 1 2 1 1 1 2
 $ Class      : Factor w/ 2 levels "no-recurrence-events",..: 2 1 2 1 2 1 1 1 1 1


,age,menopause,tumor-size,inv-nodes,node-caps,deg-malig,breast,breast-quad,irradiat,Class
,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>
1,40-49,premeno,15-19,0-2,yes,3,right,left_up,no,recurrence-events
2,50-59,ge40,15-19,0-2,no,1,right,central,no,no-recurrence-events
3,50-59,ge40,35-39,0-2,no,2,left,left_low,no,recurrence-events
4,40-49,premeno,35-39,0-2,yes,3,right,left_low,yes,no-recurrence-events
5,40-49,premeno,30-34,3-5,yes,2,left,right_up,no,recurrence-events
6,50-59,premeno,25-29,3-5,no,2,right,left_up,yes,no-recurrence-events


### Comprobación de niveles y distribución de la clase

Al importar desde ARFF, R conserva todos los posibles valores definidos en `@attribute` como niveles (`levels`) del factor, incluso si alguno no aparece en las primeras filas.

In [3]:
# Ver los niveles del atributo 'age'
cat("Niveles del atributo age:\n")
print(levels(datos_cancer$age))

# Tabla de frecuencias de la clase objetivo
cat("\nDistribución de la clase objetivo (Class):\n")
print(table(datos_cancer$Class))

Niveles del atributo age:
[1] "40-49" "50-59"

Distribución de la clase objetivo (Class):

no-recurrence-events    recurrence-events 
                   7                    3 


## 3. Exportación de un `data.frame` a formato `.arff`

### Método A: Usando `write.arff()` del paquete `foreign`
Es la forma más directa. Es **fundamental** que las columnas categóricas estén definidas como `factor` en R antes de exportar; de lo contrario, `write.arff` podría tratarlas como cadenas de texto (`string`) en lugar de atributos nominales (`{valor1, valor2}`).

In [ ]:
# Filtramos, por ejemplo, solo los casos de pacientes con menopausia 'premeno'
subconjunto_premeno <- subset(datos_cancer, menopause == "premeno")

# Exportamos el subconjunto a un nuevo archivo .arff
write.arff(subconjunto_premeno, file = "breast-cancer-premeno.arff")

# Leemos las líneas del archivo generado para inspeccionar su sintaxis
cat(readLines("breast-cancer-premeno.arff"), sep = "\n")

### Método B: Exportación manual controlada (Recomendado para prácticas desde cero)

Cuando construimos un dataset desde cero (como en el juego de *Piedra, Papel, Tijera, Lagarto, Spock*), escribir la cabecera y los datos mediante conexión de archivo nos da control total sobre el nombre de `@relation` y el formato exacto de los atributos.

In [4]:
# --- MÉTODO B: Escritura manual controlada (Recomendado para la tarea) ---

# 1. Creamos un data.frame de ejemplo con partidas de Pares o Nones (máquina aleatoria)
set.seed(42) # Semilla opcional para reproducibilidad del ejemplo
n_rondas <- 5

eleccion_usuario <- c("pares", "nones", "pares", "pares", "nones")
dedos_usuario    <- c(2, 1, 4, 3, 2)
dedos_maquina    <- sample(0:5, size = n_rondas, replace = TRUE) # Elección aleatoria (0 a 5 dedos)

# Calculamos el ganador dinámicamente según la paridad de la suma
suma_dedos <- dedos_usuario + dedos_maquina
resultado_paridad <- ifelse(suma_dedos %% 2 == 0, "pares", "nones")
ganador <- ifelse(eleccion_usuario == resultado_paridad, "usuario", "maquina")

historial_pares_nones <- data.frame(
  ronda            = 1:n_rondas,
  eleccion_usuario = eleccion_usuario,
  dedos_usuario    = dedos_usuario,
  dedos_maquina    = dedos_maquina,
  ganador          = ganador,
  stringsAsFactors = FALSE
)

# 2. Función genérica para exportar el data.frame a formato ARFF estándar
exportar_a_arff <- function(df, archivo_salida, nombre_relacion) {
  con <- file(archivo_salida, open = "wt", encoding = "UTF-8")
  
  # Cabecera @relation
  writeLines(paste0("@relation ", nombre_relacion), con)
  writeLines("", con)
  
  # Definición exacta de @attribute (adaptar a los atributos del problema)
  writeLines("@attribute ronda numeric", con)
  writeLines("@attribute eleccion_usuario {pares, nones}", con)
  writeLines("@attribute dedos_usuario numeric", con)
  writeLines("@attribute dedos_maquina numeric", con)
  writeLines("@attribute ganador {usuario, maquina}", con)
  writeLines("", con)
  
  # Sección @data
  writeLines("@data", con)
  
  # Escribimos cada fila separada por comas sin comillas ni nombres de fila
  lineas_datos <- apply(df, 1, function(fila) {
    paste(trimws(fila), collapse = ",")
  })
  writeLines(lineas_datos, con)
  
  close(con)
  cat("Archivo generado correctamente:", archivo_salida, "\n")
}

# 3. Ejecutamos la exportación
exportar_a_arff(
  df = historial_pares_nones,
  archivo_salida = "partidas_pares_nones.arff",
  nombre_relacion = "juego_pares_o_nones"
)

# 4. Comprobación: leemos el archivo recién creado con read.arff()
library(foreign)
verificacion <- read.arff("partidas_pares_nones.arff")
print(verificacion)
str(verificacion)

Archivo generado correctamente: partidas_pares_nones.arff 
  ronda eleccion_usuario dedos_usuario dedos_maquina ganador
1     1            pares             2             0 usuario
2     2            nones             1             4 usuario
3     3            pares             4             0 usuario
4     4            pares             3             0 maquina
5     5            nones             2             1 usuario
'data.frame':	5 obs. of  5 variables:
 $ ronda           : num  1 2 3 4 5
 $ eleccion_usuario: Factor w/ 2 levels "nones","pares": 2 1 2 2 1
 $ dedos_usuario   : num  2 1 4 3 2
 $ dedos_maquina   : num  0 4 0 0 1
 $ ganador         : Factor w/ 2 levels "maquina","usuario": 2 2 2 1 2
